In [9]:
# Install dependencies
%pip install anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [10]:
# Load env variables and create client
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [11]:
# Helper functions
def add_user_message(messages, text):
  user_message = {"role": "user", "content": text}
  messages.append(user_message)


def add_assistant_message(messages, text):
  assistant_message = {"role": "assistant", "content": text}
  messages.append(assistant_message)


def chat(messages, system=None, stop_sequences=None):
  params = {
    "model": "claude-sonnet-4-5",
    "max_tokens": 1000,
    "messages": messages,
    "stop_sequences": stop_sequences,
  }

  if system:
    params["system"] = system

  message = client.messages.create(**params)

  return message.content[0].text

In [37]:
import json

def generate_dataset():
  prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
    "format": "json" or "python" or "regex",
    "solution_criteria": "Key criteria for evaluating the solution",
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

  messages = []
  add_user_message(messages, prompt)
  add_assistant_message(messages, "```json")
  text = chat(messages, stop_sequences=["```"])
  return json.loads(text)

In [38]:
dataset = generate_dataset()

dataset

with open("dataset.json", "w") as f:
  json.dump(dataset, f, indent=2)

In [41]:
def run_prompt(test_case):
  """Merges the prompt and test case input, then returns the result"""
  prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
"""

  messages = []
  add_user_message(messages, prompt)
  add_assistant_message(messages, "```code")
  output = chat(messages, stop_sequences=["```"])
  return output

In [42]:
def grade_by_model(test_case, output):
  # Create evaluation prompt
  eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Criteria you should use to evaluate the solution:
<criteria>
{test_case["solution_criteria"]}
</criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
  """

  messages = []
  add_user_message(messages, eval_prompt)
  add_assistant_message(messages, "```json")
  eval_text = chat(messages, stop_sequences=["```"])

  return json.loads(eval_text)


In [43]:
import re
import ast


def validate_json(text):
  try:
    json.loads(text.strip())
    return 10
  except json.JSONDecodeError:
    return 0


def validate_python(text):
  try:
    ast.parse(text.strip())
    return 10
  except SyntaxError:
    return 0


def validate_regex(text):
  try:
    re.compile(text.strip())
    return 10
  except re.error:
    return 0


def grade_syntax(response, test_case):
  format = test_case["format"]
  if format == "json":
    return validate_json(response)
  elif format == "python":
    return validate_python(response)
  else:
    return validate_regex(response)

In [44]:
def run_test_case(test_case):
  """Calls run_prompt, then grades the result"""
  output = run_prompt(test_case)

  # TODO - Grading
  model_grade = grade_by_model(test_case, output)
  model_score = model_grade["score"]
  reasoning = model_grade["reasoning"]

  syntax_score = grade_syntax(output, test_case)

  score = (model_score + syntax_score) / 2

  return {
    "output": output,
    "test_case": test_case,
    "score": score,
    "reasoning": reasoning,
  }

In [45]:
from statistics import mean

def run_eval(dataset):
  """Loads the dataset and calls run_test_case with each case"""
  results = []

  for test_case in dataset:
    result = run_test_case(test_case)
    results.append(result)

  average_score = mean([result["score"] for result in results])
  print(f"Average score: {average_score}")

  return results

In [46]:
with open("dataset.json", "r") as f:
  dataset = json.load(f)

results = run_eval(dataset)

print(json.dumps(results, indent=2))

Average score: 8.166666666666666
[
  {
    "output": "\nimport json\n\ndef parse_arn(arn):\n    parts = arn.split(':', 5)\n    return {\n        'partition': parts[1] if len(parts) > 1 else None,\n        'service': parts[2] if len(parts) > 2 else None,\n        'region': parts[3] if len(parts) > 3 else None,\n        'account-id': parts[4] if len(parts) > 4 else None,\n        'resource': parts[5] if len(parts) > 5 else None\n    }\n\nprint(json.dumps(parse_arn(\"arn:aws:s3:us-east-1:123456789012:bucket/my-bucket\")))\n",
    "test_case": {
      "task": "Write a function that parses an AWS ARN (Amazon Resource Name) and returns a dictionary containing the partition, service, region, account-id, and resource components",
      "format": "python",
      "solution_criteria": "Function correctly splits ARN string on colons, handles ARNs with and without regions/account-ids, returns dict with keys: partition, service, region, account_id, resource"
    },
    "score": 8.0,
    "reasoning":